In [10]:
import torch
def find_tensor_sub_seq(batch_ids, sub_seq_ids):    
    first_token = sub_seq_ids[0]
    batch_size, seq_len = batch_ids.shape
    sub_seq_len = len(sub_seq_ids)
    positions = torch.full((batch_size,), seq_len, dtype=torch.int, device=batch_ids.device)
    for b in range(batch_size):
        seq = batch_ids[b]
        candidate_positions = (seq == first_token).nonzero(as_tuple=True)[0]
        for pos in candidate_positions:
            if pos + sub_seq_len <= seq_len:
                window = seq[pos:pos+sub_seq_len]
                if torch.all(window == sub_seq_ids):
                    positions[b] = pos
                    break
    return positions



input_ids = torch.Tensor([[1,2,3,4,5,6,7]])

sor_ids_tensor = torch.Tensor([2123,3412])

eor_ids_tensor = torch.Tensor([15126,71242])

sep_id = 4

print(input_ids.shape[-1])

#obtain prompt mask to locate prompt tokens
sor_positions = find_tensor_sub_seq(input_ids, sor_ids_tensor)
eor_positions = find_tensor_sub_seq(input_ids, eor_ids_tensor)

print(sor_positions)
print(eor_positions)

prompt_complete_flags = (sor_positions < eor_positions).unsqueeze(1)
seq_range = torch.arange(input_ids.size(1), device=input_ids.device).unsqueeze(0)

end_prompt_mask = (eor_positions < input_ids.shape[-1]) & (seq_range
                    <= (eor_positions + len(eor_ids_tensor) - 1)
                    .unsqueeze(1)).bool()
start_prompt_mask = (sor_positions < input_ids.shape[-1]) & (seq_range
                >= sor_positions
                .unsqueeze(1)).bool()

prompt_mask = torch.where(
    prompt_complete_flags, 
    start_prompt_mask & end_prompt_mask, 
    start_prompt_mask | end_prompt_mask
)

#obtain sep_id mask
sep_mask = input_ids == sep_id

teacher_force_mask = prompt_mask | sep_mask

teacher_force_mask

7
tensor([7], dtype=torch.int32)
tensor([7], dtype=torch.int32)


tensor([[False, False, False,  True, False, False, False]])